# Forecasting and Nowcast Workflow


In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np

from pynns import nns_arma, nns_arma_optim, nns_nowcast_panel, nns_seas, nns_var
from pynns.providers import CsvNowcastProvider

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(21)


## Monthly series


In [2]:
t = np.arange(1, 49, dtype=np.float64)
dates = [f"2020-{month:02d}" for month in range(1, 13)] + [f"2021-{month:02d}" for month in range(1, 13)] + [f"2022-{month:02d}" for month in range(1, 13)] + [f"2023-{month:02d}" for month in range(1, 13)]
revenue = 120.0 + 1.2 * t + 9.0 * np.sin(2.0 * np.pi * t / 12.0) + rng.normal(0.0, 1.5, t.size)
orders = 85.0 + 0.7 * t + 6.0 * np.sin(2.0 * np.pi * (t + 2.0) / 12.0) + rng.normal(0.0, 1.2, t.size)
activity = 50.0 + 0.4 * t + 5.0 * np.cos(2.0 * np.pi * t / 6.0) + rng.normal(0.0, 1.0, t.size)
panel = np.column_stack((revenue, orders, activity))
panel_with_missing = panel.copy()
panel_with_missing[10, 1] = np.nan
panel_with_missing[27, 2] = np.nan

print("panel shape:", panel_with_missing.shape)
print("last observed rows:")
print(panel_with_missing[-3:])


panel shape: (48, 3)
last observed rows:
[[166.973  116.1881  66.6082]
 [169.8529 121.9687  70.8091]
 [177.4491 124.202   74.7791]]


## Seasonality and ARMA


In [3]:
seas = nns_seas(revenue, modulo=[3, 6, 12], mod_only=True)
arma_lin = nns_arma(revenue, h=4, seasonal_factor=12, method="lin")
arma_both = nns_arma(revenue, h=4, seasonal_factor=12, method="both")
optim = nns_arma_optim(
    revenue,
    h=4,
    seasonal_factor=[6, 12],
    lin_only=True,
    print_trace=False,
)

print("candidate periods:", seas["periods"])
print("best period:", seas["best.period"])
print("ARMA lin forecast:", np.round(arma_lin, 2))
print("ARMA both forecast:", np.round(arma_both, 2))
print("optimized periods/method:", optim["periods"], optim["method"])
print("optimized forecast:", np.round(optim["results"], 2))


candidate periods: [ 3  6  9 12 15 18]
best period: 3
ARMA lin forecast: [183.51 186.32 191.99 189.21]
ARMA both forecast: [184.24 185.83 190.68 188.25]
optimized periods/method: [12] lin
optimized forecast: [183.22 186.03 191.7  188.92]


## VAR forecast


In [4]:
var = nns_var(
    panel_with_missing,
    h=3,
    tau=[1, 2, 3],
    dim_red_method="cor",
    naive_weights=False,
    status=False,
)
print("interpolated missing rows:")
print(var["interpolated_and_extrapolated"][[10, 27]])
print("univariate forecast:")
print(np.round(var["univariate"], 2))
print("multivariate forecast:")
print(np.round(var["multivariate"], 2))
print("ensemble forecast:")
print(np.round(var["ensemble"], 2))
print("relevant variables:")
print(var["relevant_variables"])


interpolated missing rows:
[[129.5761  96.1406  55.516 ]
 [163.7713 104.8407  60.3538]]
univariate forecast:
[[183.22 118.2   72.9 ]
 [186.03 119.87  67.23]
 [191.7  119.54  64.63]]
multivariate forecast:
[[173.02 113.83  65.31]
 [174.92 114.22  65.66]
 [177.45 124.2   74.78]]
ensemble forecast:
[[175.06 114.7   66.83]
 [177.14 115.35  65.97]
 [180.3  123.27  72.75]]
relevant variables:
[['x2_tau_0' 'x1_tau_0' 'x1_tau_0']
 ['x3_tau_0' 'x3_tau_0' 'x2_tau_0']
 ['x1_tau_1' 'x1_tau_1' 'x1_tau_1']
 ['x2_tau_2' 'x2_tau_2' 'x2_tau_2']
 ['x3_tau_3' 'x3_tau_3' 'x3_tau_3']]


## Local nowcast panel
R NNS 13.0 does not export `NNS.nowcast`; PyNNS keeps the local panel workflow.


In [5]:
monthly_payload = {
    "revenue": revenue[-24:].copy(),
    "orders": orders[-24:].copy(),
    "activity": activity[-24:].copy(),
}
nowcast = nns_nowcast_panel(
    monthly_payload,
    h=2,
    tau=12,
    dates=dates[-24:],
    dim_red_method="cor",
    naive_weights=False,
)
print("names:", nowcast["names"])
print("forecast dates:", nowcast["dates"]["forecast"])
print("ensemble forecast:")
print(np.round(nowcast["ensemble"], 2))
print("metadata:", nowcast["metadata"])


names: ['revenue', 'orders', 'activity']
forecast dates: ['2024-01', '2024-02']
ensemble forecast:
[[173.35 120.88  73.69]
 [173.47 123.53  71.78]]
metadata: {'source': 'user_panel', 'freq': 'monthly', 'tau': 12, 'dim_red_method': 'cor', 'naive_weights': False}


## CSV provider


In [6]:
with TemporaryDirectory() as tmp:
    path = Path(tmp) / "monthly_panel.csv"
    rows = ["date,revenue,orders,activity"]
    for date, row in zip(dates[-18:], panel[-18:], strict=True):
        rows.append(f"{date},{row[0]:.6f},{row[1]:.6f},{row[2]:.6f}")
    path.write_text("\n".join(rows) + "\n", encoding="utf-8")

    provider = CsvNowcastProvider(path)
    payload = provider.fetch((), dates[-18])
    csv_result = nns_nowcast_panel(payload["series"], h=1, tau=6, dates=payload["dates"])

metadata = dict(payload["metadata"])
metadata["path"] = "<temporary CSV>"
print("provider metadata:", metadata)
print("provider dates:", payload["dates"][:3], "...", payload["dates"][-3:])
print("one-step CSV nowcast:", np.round(csv_result["ensemble"], 2))


provider metadata: {'provider': 'csv', 'path': '<temporary CSV>', 'date_column': 'date', 'series_columns': ['revenue', 'orders', 'activity']}
provider dates: ['2022-07', '2022-08', '2022-09'] ... ['2023-10', '2023-11', '2023-12']
one-step CSV nowcast: [[172.2  122.28  68.33]]
